Project Structure
- Setup Chunking and processing steps for inputs (PDFs, docx, txt)
- Setup PGVector Store (VS) + postgresql => Done
- Using LLMs (OpenAI, Gemini,...) to query vectors from VS

# Create PostgreDB 

In [3]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

In [4]:
import psycopg2

db_name = os.environ['POSTGRES_DB']
host = "localhost"
password = os.environ['POSTGRES_PASSWORD']
port = "5432"
user = os.environ['POSTGRES_USER']
# conn = psycopg2.connect(connection_string)
conn = psycopg2.connect(
    dbname=db_name,
    host=host,
    password=password,
    port=port,
    user=user,
)
conn.autocommit = True

# with conn.cursor() as c:
#     c.execute(f"DROP DATABASE IF EXISTS {db_name}")
#     c.execute(f"CREATE DATABASE {db_name}")

ModuleNotFoundError: No module named 'psycopg2'

In [5]:
print(conn.status)
print(conn.get_dsn_parameters())
with conn.cursor() as cursor:
    cursor.execute("SELECT 1")
    print("Connection is active")

1
{'user': 'admin', 'channel_binding': 'prefer', 'dbname': 'rag_db', 'host': 'localhost', 'port': '5432', 'options': '', 'sslmode': 'prefer', 'sslcompression': '0', 'sslcertmode': 'allow', 'sslsni': '1', 'ssl_min_protocol_version': 'TLSv1.2', 'gssencmode': 'prefer', 'krbsrvname': 'postgres', 'gssdelegation': '0', 'target_session_attrs': 'any', 'load_balance_hosts': 'disable'}
Connection is active


# Checking file input

In [1]:
from pydantic import BaseModel, Field, field_validator
from pydantic_ai import Agent, RunContext
from uuid import UUID
from chonkie import TokenChunker, SemanticChunker

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import PyPDF2
from chonkie import SemanticChunker
from sentence_transformers import SentenceTransformer


def extract_text_from_pdf(pdf_path):
    """
    Extract text from PDF file using PyPDF2.

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        str: Extracted text from all pages
    """
    text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)

        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text += page.extract_text() + "\n"

    return text


def chunk_with_semantic_chunker(text, chunk_size=512, similarity_threshold=0.7, embedding_model=None):
    """
    Chunk text using Chonkie's SemanticChunker with custom embedding model.

    Args:
        text (str): Text to chunk
        chunk_size (int): Maximum tokens per chunk
        similarity_threshold (float): Similarity threshold for semantic chunking
        embedding_model: Custom embedding model (SentenceTransformer or similar)

    Returns:
        list: List of chunks
    """
    if embedding_model:
        chunker = SemanticChunker(
            chunk_size=chunk_size,
            similarity_threshold=similarity_threshold,
            embedding_model=embedding_model
        )
    else:
        # Use default embedding model
        chunker = SemanticChunker(
            chunk_size=chunk_size,
            similarity_threshold=similarity_threshold
        )

    chunks = chunker.chunk(text)
    return chunks


def save_chunks_to_file(chunks, output_path, chunker_type):
    """
    Save chunks to a text file.

    Args:
        chunks (list): List of chunks
        output_path (str): Output file path
        chunker_type (str): Type of chunker used
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(f"Chunks created using {chunker_type}\n")
        f.write("=" * 50 + "\n\n")

        for i, chunk in enumerate(chunks, 1):
            f.write(f"Chunk {i}:\n")
            f.write("-" * 20 + "\n")
            f.write(f"{chunk.text}\n\n")
            f.write(f"Tokens: {chunk.token_count}\n")
            if hasattr(chunk, 'start_index'):
                f.write(f"Start Index: {chunk.start_index}\n")
            if hasattr(chunk, 'end_index'):
                f.write(f"End Index: {chunk.end_index}\n")
            f.write("\n" + "="*50 + "\n\n")

/usr/local/Caskroom/miniconda/base/envs/project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
print("\nStep 1: Loading documents")
# texts = extract_text_from_pdf(r'D:\Project\rag_with_llama\docs\llama2.pdf')
texts = extract_text_from_pdf(r'../docs/llama2.pdf')

# Create output directory
output_dir = "chunked_output"
os.makedirs(output_dir, exist_ok=True)

print("\nStep 2: Chunking with SemanticChunker")
semantic_chunks = chunk_with_semantic_chunker(
    texts,
    chunk_size=512,
    similarity_threshold=0.7
    )


Step 1: Loading documents

Step 2: Chunking with SemanticChunker


/usr/local/Caskroom/miniconda/base/envs/project/lib/python3.12/site-packages/chonkie/embeddings/auto.py:87: UserWarning: Failed to load minishlab/potion-base-32M with Model2VecEmbeddings: model2vec is not available. Please install it via `pip install chonkie[model2vec]`
Falling back to loading default provider model.
  warnings.warn(
/usr/local/Caskroom/miniconda/base/envs/project/lib/python3.12/site-packages/chonkie/embeddings/auto.py:95: UserWarning: Failed to load the default model for Model2VecEmbeddings: model2vec is not available. Please install it via `pip install chonkie[model2vec]`
Falling back to SentenceTransformerEmbeddings.
  warnings.warn(


In [14]:
semantic_chunks[0]

Chunk(text='Llama 2 : Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗Louis Martin†Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
', token_count=43, start_index=0, end_index=167)

In [ ]:
from typing import List, Dict, Any, Optional

class EmbeddingGenerator:
    """Generate embeddings using SentenceTransformers."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize embedding generator.

        Args:
            model_name: SentenceTransformer model name
        """
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

    def embed_text(self, text: str) -> List[float]:
        """
        Generate embedding for a single text.
        Args:
            text: Input text
        Returns:
            List of embedding values
        """
        embedding = self.model.encode(text)
        return embedding.tolist()

    def embed_batch(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for multiple texts.
        Args:
            texts: List of input texts
        Returns:
            List of embedding lists
        """
        embeddings = self.model.encode(texts)
        return [emb.tolist() for emb in embeddings]

embedding_model = 'all-MiniLM-L6-v2'
embedding_generator = EmbeddingGenerator(embedding_model)

In [16]:
embedded_txt = embedding_generator.embed_batch(semantic_chunks)

In [17]:
len(embedded_txt)

1332

In [1]:
import psycopg2
from pgvector.psycopg2 import register_vector  # Missing import

In [ ]:
# -- First, turn off the pager to avoid the 'more' error
# \pset pager off

# -- Now run your queries (notice the semicolon at the end)
# SELECT version();

# SELECT 2+2 AS result;

# SELECT NOW();

# SELECT 'Hello PostgreSQL!' AS message;

In [ ]:
# import sentence_transformers
# model = sentence_transformers.SentenceTransformer('all-MiniLM-L6-v2')
# model.save('./embedded_model/all-MiniLM-L6-v2')
# model_test = sentence_transformers.SentenceTransformer('./embedded_model/all-MiniLM-L6-v2')

# Test fulll flow

In [14]:
%reload_ext autoreload
%autoreload 2

In [1]:
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any

from fastapi import FastAPI, File, UploadFile, HTTPException, Form
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

import sys
sys.path.append('../document_processing')  # Go up one level, then into folder

import sys
sys.path.append('../docs')  # Go up one level, then into folder

# Your existing components - unchanged
from embed_chunks_to_db import ChunkEmbeddingPipeline, EmbeddingGenerator, VectorStore
import google.generativeai as genai

/opt/miniconda3/envs/longnv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
db_params = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': os.getenv('DB_PORT', '5432'),
    'dbname': os.getenv('DB_NAME', 'rag_db'),
    'user': os.getenv('DB_USER', 'admin'),
    'password': os.getenv('DB_PASSWORD', 'admin')
}

table_name = 'document_chunks'

# Don't initialize models at startup
pipeline = None
def get_pipeline():
    global pipeline
    if pipeline is None:
        pipeline = ChunkEmbeddingPipeline(
                    db_params=db_params,
                    embedding_model='all-MiniLM-L6-v2',
                    table_name=table_name
                )
    return pipeline

pipeline = get_pipeline()
file = pipeline.extract_text_from_pdf(r'../docs/llama2.pdf')
chunks = pipeline.chunk_text(file)

In [14]:
def test_pgvector_connection():
    """Validate pgvector integration at the connection level"""
    vector_store = VectorStore(db_params, 'document_chunks')
    
    try:
        conn = vector_store._get_connection()
        with conn.cursor() as cur:
            # Test 1: Confirm pgvector types are registered
            cur.execute("SELECT '[1,2,3]'::vector;")
            result = cur.fetchone()[0]
            print(f"✓ Vector type working: {result}")
            
            # Test 2: Verify cosine distance operator
            cur.execute("SELECT '[1,0,0]'::vector <=> '[0,1,0]'::vector;")
            distance = cur.fetchone()[0]
            print(f"✓ Cosine distance: {distance:.3f}")
            
            # Test 3: Table schema validation
            cur.execute(f"""
                SELECT column_name, data_type 
                FROM information_schema.columns 
                WHERE table_name = '{vector_store.table_name}' 
                AND column_name = 'embedding'
            """)
            schema = cur.fetchone()
            print(f"✓ Embedding column: {schema}")
            
        conn.close()
        return True
        
    except Exception as e:
        print(f"✗ pgvector connection failed: {e}")
        return False
    
test_pgvector_connection()

✓ Vector type working: [1. 2. 3.]
✓ Cosine distance: 1.000
✓ Embedding column: ('embedding', 'USER-DEFINED')


True

In [15]:
from dataclasses import dataclass
from typing import List, Dict, Any, Optional


@dataclass
class Chunk:
    """Chunk data structure to match your existing interface."""
    id: str
    document_id: str
    text: str
    embedding: List[float]
    metadata: Optional[Dict] = None

In [16]:
# Generate embeddings in batch
print("Generating embeddings...")
embeddings = pipeline.embedding_generator.embed_batch([chunk.text for chunk in chunks])

Generating embeddings...


In [17]:
chunk_size = 512
similarity_threshold = 0.7
filename = 'llama2.pdf'
file_type = 'pdf'
file_size = len(file)
document_id = str(uuid.uuid4())
metadata = {'source': 'llama2.pdf'}

# Create Chunk objects using your interface
chunk_objects = []
for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    chunk_metadata = {
        'chunk_index': i,
        'token_count': chunk.token_count,
        'start_index': getattr(chunk, 'start_index', None),
        'end_index': getattr(chunk, 'end_index', None),
        'chunk_size': chunk_size,
        'similarity_threshold': similarity_threshold,
        'embedding_model': pipeline.embedding_generator.model_name,
        'embedding_dimension': len(embedding),
        'filename': filename,
        'file_type': file_type,
        'file_size': file_size
    }

    # Add any additional metadata
    if metadata:
        chunk_metadata.update(metadata)

    chunk_obj = Chunk(
        id=str(uuid.uuid4()),
        document_id=document_id,
        text=chunk.text,
        embedding=embedding,
        metadata=chunk_metadata
    )
    chunk_objects.append(chunk_obj)

In [18]:
# Use your add_chunks method with pgvector
print("Inserting chunks into database using pgvector...")
pipeline.vector_store.add_chunks(chunk_objects)

Inserting chunks into database using pgvector...


In [19]:
test_results = pipeline.search_documents('text', limit=1, threshold=0.1)
assert len(test_results) > 0, "Insert succeeded but search failed"
print(f"✓ Search operational: {len(test_results)} results")

✓ Search operational: 1 results


In [ ]:
def fast_reset_vector_store(vector_store):
    """Two-step reset: TRUNCATE then DROP for optimal performance"""
    conn = vector_store._get_connection()
    
    try:
        with conn.cursor() as cur:
            # Step 1: Instant data removal (no WAL overhead)
            cur.execute(f"TRUNCATE TABLE {vector_store.table_name} CASCADE;")
            print("✓ Data truncated instantly")
            
            # Step 2: Clean schema removal
            cur.execute(f"DROP TABLE {vector_store.table_name} CASCADE;")
            print("✓ Table schema dropped")
            
        conn.commit()
        
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

# # Usage
# fast_reset_vector_store(pipeline.vector_store)

✓ Data truncated instantly
✓ Table schema dropped


# Check inserting 

In [44]:
# Verify success
conn = pipeline.vector_store._get_connection()
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {pipeline.vector_store.table_name}")
    count_after = cur.fetchone()[0]
    
    # Validate embedding integrity
    cur.execute(f"""
        SELECT count(*)
        FROM {pipeline.vector_store.table_name} 
    """)
    
    results = cur.fetchall()

UndefinedTable: relation "document_chunks" does not exist
LINE 1: SELECT COUNT(*) FROM document_chunks
                             ^


In [43]:
results[0][0]

2740

In [21]:
results[0]

('0cef73c0-a5ea-4b45-93f7-152110c773a0',
 '62fea6ed-85a6-464f-b0fd-5ec2402d0dd8',
 'Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\n',
 array([ 7.12140650e-03,  1.58904716e-01, -4.11183424e-02, -1.60896499e-02,
        -1.19244307e-01,  1.66793726e-02,  7.66922235e-02,  2.09896639e-03,
        -7.99249765e-03, -3.11004985e-02,  6.69134455e-03, -1.19025603e-01,
         9.55456793e-02, -4.00688797e-02,  2.43884884e-03,  2.39558239e-02,
        -2.64738058e-03,  2.02801265e-02,  7.83016905e-03, -7.15690851e-02,
        -2.38060020e-02, -1.06638998e-01, -9.48443916e-03,  2.73222420e-02,
        -6.30895793e-02, -4.59778449e-03,  1.97286680e-02,  1.22988073e-03,
         2.02850271e-02, -3.98318060e-02, -3.29108462e-02,  1.30037352e-01,
         7.59432688e-02, -2.78772581e-02, -1.24753239e-02,  8.16547722e-02,
        -5.94117120e-02,  4.51860111e-03,  1.69892292e-02, -4.73478902e-03,
        -2.29553226e-02,  7.49923289e-03, -2.45867502e-02, -6.

In [46]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('./deployment/.env')

False

In [34]:
# embedding_model = './embedded_model/all-MiniLM-L6-v2'
# embedding_generator = EmbeddingGenerator(embedding_model)
query = "What is Llama 2?"
results = pipeline.search_documents(
    query=query,
    limit=100,
    threshold=0.5
)

# Step 2: Build context for LLM
context = "\n\n".join([f"[Context {i+1}]: {r['text']}" for i, r in enumerate(results)])

# Step 3: Generate response with Gemini (if available)
gemini_key = os.getenv('GOOGLE_API_KEY')
genai.configure(api_key=gemini_key)
if gemini_key:
    model = genai.GenerativeModel('gemini-2.5-flash')
    prompt = f"""Answer based on this context:
            {context}
            Question: {query}
            Answer:"""
    
    response = model.generate_content(prompt)
    answer = response.text

E0000 00:00:1758732769.428342 1368407 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [35]:
answer

'Llama 2 is an updated version of Llama 1, trained on a new mix of publicly available data. It is a new technology and a Large Language Model (LLM) that was pretrained on 2 trillion tokens of data from publicly available sources. Llama 2 carries potential risks with its use.'

# Test with Pydantic AI

In [6]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../deployment/.env')

True

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from pydantic import BaseModel, Field
from typing import Dict, Any, List, Optional

class ResponseWithMetadata(BaseModel):
    content: str = Field(description="Main response")
    confidence: Optional[float] = Field(None, ge=0, le=1)
    word_count: Optional[int] = None
    complexity_level: Optional[str] = None
    custom_metadata: Dict[str, Any] = Field(default_factory=dict)

# Create agent with structured response
agent = Agent(model)

provider = GoogleProvider(api_key=os.getenv('GOOGLE_API_KEY'))
model = GoogleModel('gemini-2.5-flash', provider=provider)
agent = Agent(model, result_type=ResponseWithMetadata)

In [18]:
answer

AgentRunResult(output='**NLTK (Natural Language Toolkit)** is a powerful, open-source library written in **Python** that is widely used for working with human language data. In the context of text processing, it serves as a comprehensive platform for building Python programs to analyze, process, and understand natural language.\n\nThink of it as a Swiss Army knife for Natural Language Processing (NLP) tasks, especially for educational purposes, research, and rapid prototyping.\n\nHere\'s a breakdown of what NLTK is and its role in text processing:\n\n### What is NLTK?\n\n*   **Python Library:** It\'s a collection of modules and functions for Python.\n*   **NLP Focus:** Specifically designed for tasks related to Natural Language Processing, which is a subfield of artificial intelligence that deals with the interaction between computers and human (natural) languages.\n*   **Comprehensive:** It provides a vast array of algorithms and data sets (corpora) that support almost all facets of N

In [10]:
question = 'What is NLTK in context of text processing?'
answer = await agent.run(question)

# Logfire test

In [1]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../deployment/.env')

True

In [ ]:
import logfire

logfire_key = os.getenv('LOGFIRE_WRITE_TOKEN')
logfire.configure(token=logfire_key)

# Different log levels
logfire.trace("Trace level message")
logfire.debug("Debug message with {var}", var="value")
logfire.info("Hello, {name}!", name="world")
logfire.notice("Notice level")
logfire.warn("Warning message")
logfire.error("Error occurred")
logfire.fatal("Fatal error")

11:17:30.226 Hello, world!
11:17:30.226 Notice level
11:17:30.227 Warning message
11:17:30.227 Error occurred
11:17:30.227 Fatal error


Logfire project URL: ]8;id=223067;https://logfire-us.pydantic.dev/adrien-2025/rag-demo\https://logfire-us.pydantic.dev/adrien-2025/rag-demo]8;;\
Currently retrying 1 failed export(s)


In [5]:
import logfire
import time

logfire_key = os.getenv('LOGFIRE_WRITE_TOKEN')
logfire.configure(token=logfire_key)

# Top-level span
with logfire.span("Process Order", order_id="ORD-12345"):
    logfire.info("Starting order processing")
    
    # First child span
    with logfire.span("Validate Order"):
        logfire.debug("Checking order items")
        time.sleep(0.1)
        logfire.info("Order validation: {status}", status="passed")
    
    # Second child span
    with logfire.span("Calculate Total"):
        logfire.debug("Item count: {count}", count=3)
        time.sleep(0.05)
        total = 99.99
        logfire.info("Total calculated: ${amount}", amount=total)
    
    # Third child span
    with logfire.span("Process Payment"):
        logfire.info("Charging card ending in {last4}", last4="4242")
        time.sleep(0.2)
        logfire.info("Payment successful")
    
    logfire.info("Order completed successfully")

13:46:29.744 Process Order
13:46:29.744   Starting order processing
13:46:29.744   Validate Order
13:46:29.850     Order validation: passed
13:46:29.850   Calculate Total
13:46:29.906     Total calculated: $99.99
13:46:29.906   Process Payment
13:46:29.907     Charging card ending in 4242
13:46:30.112     Payment successful
13:46:30.113   Order completed successfully


Logfire project URL: ]8;id=463429;https://logfire-us.pydantic.dev/adrien-2025/rag-demo\https://logfire-us.pydantic.dev/adrien-2025/rag-demo]8;;\


# Test with ReRank

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../deployment/.env')

import json
import uuid
import asyncpg
import numpy as np
import PyPDF2
from pathlib import Path
import asyncio

# Disable tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Import your existing chunking functions
sys.path.append('/Users/longnv/Coding/rag_llama_index/document_processing')

from sentence_transformers import SentenceTransformer

from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from chunk_pdf_with_chonkie import process_document


@dataclass
class Chunk:
    """Chunk data structure to match your existing interface."""
    id: str
    document_id: str
    text: str
    embedding: List[float]
    metadata: Optional[Dict] = None


class EmbeddingGenerator:
    """Generate embeddings using SentenceTransformers."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize embedding generator.

        Args:
            model_name: SentenceTransformer model name
        """
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

    def embed_text(self, text: str) -> List[float]:
        """
        Generate embedding for a single text.
        Args:
            text: Input text
        Returns:
            List of embedding values
        """
        embedding = self.model.encode(text)
        return embedding.tolist()

    def embed_batch(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for multiple texts.
        Args:
            texts: List of input texts
        Returns:
            List of embedding lists
        """
        embeddings = self.model.encode(texts)
        return [emb.tolist() for emb in embeddings]


class VectorStore:
    """Vector store using pgvector for efficient similarity search."""

    def __init__(self, connection_params: Dict[str, str], table_name: str = "chunks"):
        """
        Initialize vector store with pgvector support.
        Args:
            connection_params: Database connection parameters
            table_name: Name of the chunks table
        """
        self.connection_params = connection_params
        self.table_name = table_name
        self.connection_string = self._build_connection_string()
        self._initialized = False

    def _build_connection_string(self) -> str:
        """Build asyncpg connection string from parameters."""
        return f"postgresql://{self.connection_params['user']}:{self.connection_params['password']}@{self.connection_params['host']}:{self.connection_params['port']}/{self.connection_params['dbname']}"

    async def _get_connection(self):
        """Get database connection with pgvector support."""
        conn = await asyncpg.connect(self.connection_string)
        # Enable pgvector extension
        await conn.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        return conn

    async def _initialize_database(self):
        """Initialize database with pgvector extension and table."""
        try:
            if self._initialized:
                return

            conn = await self._get_connection()

            # Create table with proper vector column
            # Assuming 384-dimensional embeddings for all-MiniLM-L6-v2
            # Adjust dimension based on your model
            await conn.execute(f"""
            CREATE TABLE IF NOT EXISTS {self.table_name} (
                id TEXT PRIMARY KEY,
                document_id TEXT NOT NULL,
                text TEXT NOT NULL,
                embedding vector(384),  -- Adjust dimension as needed
                metadata JSONB,
                created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
            );
            """)

            # Create index for similarity search
            await conn.execute(f"""
            CREATE INDEX IF NOT EXISTS {self.table_name}_embedding_idx
            ON {self.table_name} USING ivfflat (embedding vector_cosine_ops)
            WITH (lists = 100);
            """)

            # Create index on document_id for filtering
            await conn.execute(f"""
            CREATE INDEX IF NOT EXISTS {self.table_name}_document_id_idx
            ON {self.table_name} (document_id);
            """)

            await conn.close()
            self._initialized = True
            print(f"Database initialized with table: {self.table_name}")

        except Exception as e:
            print(f"Error initializing database: {e}")
            raise

    async def add_chunks(self, chunks: List[Chunk], batch_size: int = 100):
        """Add chunks to vector store using batch insert for efficiency."""
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()

            # Prepare data for batch insert
            chunk_data = []
            for chunk in chunks:
                # Convert embedding list to proper pgvector format
                embedding_str = '[' + ','.join(map(str, chunk.embedding)) + ']'

                chunk_data.append((
                    chunk.id,
                    chunk.document_id,
                    chunk.text,
                    embedding_str,
                    json.dumps(
                        chunk.metadata) if chunk.metadata else json.dumps({})
                ))

            # Use asyncpg's executemany for efficient batch insert
            insert_sql = f"""
            INSERT INTO {self.table_name} (id, document_id, text, embedding, metadata)
            VALUES ($1, $2, $3, $4, $5)
            ON CONFLICT (id) DO UPDATE SET
                document_id = EXCLUDED.document_id,
                text = EXCLUDED.text,
                embedding = EXCLUDED.embedding,
                metadata = EXCLUDED.metadata;
            """

            # Process in batches
            for i in range(0, len(chunk_data), batch_size):
                batch = chunk_data[i:i + batch_size]
                await conn.executemany(insert_sql, batch)

            await conn.close()
            print(f"Added {len(chunks)} chunks to vector store")

        except Exception as e:
            print(f"Error adding chunks: {e}")
            raise

    async def search_similar_chunks(self, query_embedding: List[float],
                                    limit: int = 5, threshold: float = 0.7,
                                    document_ids: Optional[List[str]] = None) -> List[Dict]:
        """
        Search for similar chunks using pgvector cosine similarity.

        Args:
            query_embedding: Query embedding vector
            limit: Maximum number of results
            threshold: Similarity threshold (0-1, higher = more similar)
            document_ids: Optional list of document IDs to filter by

        Returns:
            List of similar chunks with metadata
        """
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()

            # Build query with optional document filtering
            base_query = f"""
                SELECT
                    id,
                    text,
                    metadata,
                    document_id,
                    (1 - (embedding <=> $1)) as similarity
                FROM {self.table_name}
                WHERE (1 - (embedding <=> $1)) >= $2
            """

            # Convert query embedding to proper pgvector format
            query_embedding_str = '[' + ','.join(map(str, query_embedding)) + ']'
            params = [query_embedding_str, threshold]

            if document_ids:
                base_query += " AND document_id = ANY($3)"
                params.append(document_ids)
                base_query += """
                    ORDER BY embedding <=> $1
                    LIMIT $4
                """
                params.append(limit)
            else:
                base_query += """
                    ORDER BY embedding <=> $1
                    LIMIT $3
                """
                params.append(limit)

            rows = await conn.fetch(base_query, *params)

            results = []
            for row in rows:
                results.append({
                    'chunk_id': row['id'],
                    'text': row['text'],
                    'metadata': row['metadata'] if isinstance(row['metadata'], (dict, type(None))) else json.loads(row['metadata']),
                    'document_id': row['document_id'],
                    'similarity': float(row['similarity'])
                })

            await conn.close()
            return results

        except Exception as e:
            print(f"Error searching chunks: {e}")
            raise

    async def delete_document_chunks(self, document_id: str) -> int:
        """Delete all chunks for a specific document."""
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()
            result = await conn.execute(
                f"DELETE FROM {self.table_name} WHERE document_id = $1", document_id)
            deleted_count = int(result.split()[-1]) if result else 0
            await conn.close()
            print(
                f"Deleted {deleted_count} chunks for document: {document_id}")
            return deleted_count
        except Exception as e:
            print(f"Error deleting document chunks: {e}")
            raise

    async def get_collection_stats(self) -> Dict[str, Any]:
        """Get statistics about the vector store."""
        try:
            if not self._initialized:
                await self._initialize_database()

            conn = await self._get_connection()
            row = await conn.fetchrow(f"""
            SELECT
                COUNT(*) as total_chunks,
                COUNT(DISTINCT document_id) as total_documents,
                AVG(LENGTH(text)) as avg_text_length,
                MIN(created_at) as earliest_chunk,
                MAX(created_at) as latest_chunk
            FROM {self.table_name}
            """)

            stats = {
                'total_chunks': row['total_chunks'],
                'total_documents': row['total_documents'],
                'avg_text_length': float(row['avg_text_length']) if row['avg_text_length'] else 0,
                'earliest_chunk': row['earliest_chunk'].isoformat() if row['earliest_chunk'] else None,
                'latest_chunk': row['latest_chunk'].isoformat() if row['latest_chunk'] else None
            }

            await conn.close()
            return stats
        except Exception as e:
            print(f"Error getting stats: {e}")
            raise


class ChunkEmbeddingPipeline:
    """Complete pipeline for chunking documents and storing embeddings."""

    def __init__(self, db_params: Dict[str, str], embedding_model: str,
                 table_name: str):
        """
        Initialize the pipeline.
        Args:
            db_params: Database connection parameters
            embedding_model: SentenceTransformer model name
            table_name: Name of the chunks table
        """
        self.embedding_generator = EmbeddingGenerator(embedding_model)
        self.vector_store = VectorStore(db_params, table_name)

    async def process_document(self, file_path: str, chunk_size: int = 512,
                               similarity_threshold: float = 0.5,
                               document_id: str = None, metadata: Dict = None) -> str:
        """
        Process a document: chunk using imported function, then embed and store.

        Args:
            file_path: Path to the document file
            chunk_size: Maximum tokens per chunk
            similarity_threshold: Similarity threshold for chunking
            document_id: Optional document ID (if None, will generate one)
            metadata: Additional metadata for the document

        Returns:
            Document ID
        """
        file_path = Path(file_path)
        filename = file_path.name
        file_type = file_path.suffix.lower().replace('.', '')
        file_size = file_path.stat().st_size

        # Generate document ID if not provided
        if document_id is None:
            document_id = str(uuid.uuid4())

        print(f"Processing document: {filename} (ID: {document_id})")

        # Use the imported process_document function for chunking with page numbers
        chunks = process_document(
            file_path=str(file_path),
            chunk_size=chunk_size,
            similarity_threshold=similarity_threshold,
            embedding_model=self.embedding_generator.model_name
        )

        print(f"Created {len(chunks)} chunks using imported chunking function")

        # Prepare chunks for embedding
        chunk_texts = [chunk.text for chunk in chunks]

        # Generate embeddings in batch
        print("Generating embeddings...")
        embeddings = self.embedding_generator.embed_batch(chunk_texts)

        # Create Chunk objects for database storage
        chunk_objects = []
        for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
            # Extract page number from chunk (set by imported process_document function)
            page_number = getattr(chunk, 'page_number', 1)

            chunk_metadata = {
                'chunk_index': i,
                'token_count': chunk.token_count,
                'start_index': getattr(chunk, 'start_index', None),
                'end_index': getattr(chunk, 'end_index', None),
                'page_number': page_number,
                'chunk_size': chunk_size,
                'similarity_threshold': similarity_threshold,
                'embedding_model': self.embedding_generator.model_name,
                'embedding_dimension': len(embedding),
                'filename': filename,
                'file_type': file_type,
                'file_size': file_size
            }

            # Add any additional metadata
            if metadata:
                chunk_metadata.update(metadata)

            chunk_obj = Chunk(
                id=str(uuid.uuid4()),
                document_id=document_id,
                text=chunk.text,
                embedding=embedding,
                metadata=chunk_metadata
            )
            chunk_objects.append(chunk_obj)

        # Store in database using pgvector
        print("Inserting chunks into database using pgvector...")
        await self.vector_store.add_chunks(chunk_objects)

        print(
            f"Successfully processed {filename} -> Document ID: {document_id}")
        return document_id

    async def search_documents(self, query: str, limit: int = 5, threshold: float = 0.7,
                               document_ids: Optional[List[str]] = None) -> List[Dict]:
        """
        Search for relevant document chunks using pgvector.
        Args:
            query: Search query
            limit: Maximum number of results
            threshold: Similarity threshold
            document_ids: Optional list of document IDs to filter by

        Returns:
            List of relevant chunks
        """
        query_embedding = self.embedding_generator.embed_text(query)
        return await self.vector_store.search_similar_chunks(
            query_embedding, limit, threshold, document_ids
        )

    async def delete_document(self, document_id: str) -> int:
        """Delete all chunks for a document."""
        return await self.vector_store.delete_document_chunks(document_id)

    async def get_stats(self) -> Dict[str, Any]:
        """Get vector store statistics."""
        return await self.vector_store.get_collection_stats()


embedding_model = 'all-MiniLM-L6-v2'
embedding_generator = EmbeddingGenerator(embedding_model)

In [ ]:
import logfire
import google.generativeai as genai
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider
from file_validator import FileValidator, FileValidationConfig
from models.models import QueryRequest, UploadResponse, RAGResponse, SimpleRAGResponse, RAGSource, RAGResponseMetadata

# Constants
DEFAULT_TABLE_NAME = "document_chunks"
DEFAULT_EMBEDDING_MODEL = "all-MiniLM-L6-v2"
DEFAULT_CHUNKING_SIMILARITY = 0.5
ALLOWED_CONTENT_TYPES = [
    'application/pdf', 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'text/plain']

# Global configuration

class AppConfig:
    def __init__(self):
        # Initialize logfire with token from environment
        logfire_token = os.getenv('LOGFIRE_WRITE_TOKEN')
        if logfire_token:
            logfire.configure(token=logfire_token)
            print("✓ Logfire configured successfully")
        else:
            print("⚠️ LOGFIRE_WRITE_TOKEN not found, using default configuration")
            logfire.configure()

        self.db_params = {
            'host': os.getenv('DB_HOST', 'localhost'),
            'port': os.getenv('DB_PORT', '5432'),
            'dbname': os.getenv('POSTGRES_DB', 'rag_db'),
            'user': os.getenv('POSTGRES_USER', 'admin'),
            'password': os.getenv('POSTGRES_PASSWORD', 'admin')
        }
        self.file_validator = FileValidator(FileValidationConfig())
        self.agent = self._configure_pydantic_ai()
        self.pipeline = None
        self.reranker = None  # Lazy initialization to avoid startup overhead

    def _configure_pydantic_ai(self):
        gemini_key = os.getenv('GOOGLE_API_KEY')
        if gemini_key:
            try:
                # Configure Pydantic AI Agent with GoogleProvider
                provider = GoogleProvider(api_key=gemini_key)
                model = GoogleModel('gemini-2.5-flash', provider=provider)

                # Create agent with system prompt and output type
                agent = Agent(
                    model,
                    output_type=SimpleRAGResponse,
                    system_prompt="""You are a helpful RAG (Retrieval-Augmented Generation) assistant.
                    Based on the provided context from document search, provide comprehensive answers to user questions.

                    Instructions:
                    - Answer directly and accurately based on the provided context
                    - If the context doesn't fully answer the question, clearly state what information is available
                    - Cite specific sources when making claims, including page numbers when available (e.g., "according to Source 1, Page 5")
                    - Be concise but thorough
                    - Provide a confidence score (0-1) based on how well the context answers the question

                    Respond with:
                    - answer: Your comprehensive response with page references
                    - confidence: Float between 0-1 indicating confidence in the answer
                    - word_count: Number of words in your answer
                    - sources_used: Number of sources used (will be provided)
                    - metadata: Any additional relevant information"""
                )

                # Test the agent with a simple query
                print("✓ Pydantic AI Agent configured successfully")
                return agent
            except Exception as e:
                print(f"❌ Pydantic AI configuration failed: {e}")
                # Fallback to direct genai for backward compatibility
                try:
                    genai.configure(api_key=gemini_key)
                    print("✓ Fallback to direct Gemini API")
                except Exception as fallback_error:
                    print(f"❌ Gemini fallback also failed: {fallback_error}")
        return None

config = AppConfig()

✓ Logfire configured successfully
✓ Pydantic AI Agent configured successfully


Logfire project URL: ]8;id=776459;https://logfire-us.pydantic.dev/adrien-2025/rag-demo\https://logfire-us.pydantic.dev/adrien-2025/rag-demo]8;;\


In [ ]:
# Function test
async def get_pipeline(table_name: str = DEFAULT_TABLE_NAME):
    if config.pipeline is None or config.pipeline.vector_store.table_name != table_name:
        config.pipeline = ChunkEmbeddingPipeline(
            db_params=config.db_params,
            embedding_model=DEFAULT_EMBEDDING_MODEL,
            table_name=table_name
        )
        # Initialize the database for the new pipeline
        await config.pipeline.vector_store._initialize_database()
    return config.pipeline

pipeline = await get_pipeline(table_name)

table_name = 'test1'
initial_limit = 20
threshold = 0.5 
document_ids = None
query = 'What is NLTK?'

results = await pipeline.search_documents(
    query=query,
    limit=initial_limit,
    threshold=threshold,
    document_ids=document_ids
)

In [16]:
len(results)

19

In [25]:
results[0]

{'chunk_id': '6250bf1e-e76e-4bb9-b33b-d05eb07aa77c',
 'text': 'You should now be equipped to work with large datasets, to create robust models of\nlinguistic phenomena, and to extend them into components for practical language\ntechnologies. We hope that the Natural Language Toolkit (NLTK) has served to open\nup the exciting endeavor of practical natural language processing to a broader audience\nthan before.\nIn spite of all that has come before, language presents us with far more than a temporary\n',
 'metadata': {'filename': 'NLTK.pdf',
  'end_index': 1003848,
  'file_size': 3580937,
  'file_type': 'pdf',
  'chunk_size': 768,
  'chunk_index': 5268,
  'page_number': 463,
  'start_index': 1003412,
  'token_count': 83,
  'content_type': 'application/pdf',
  'embedding_model': 'all-MiniLM-L6-v2',
  'upload_timestamp': '139785087795402850',
  'validation_passed': True,
  'embedding_dimension': 384,
  'similarity_threshold': 0.5},
 'document_id': 'b715603a-d540-4474-87dc-62f26a56436d',
 '

In [22]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize the text (simple word splitting)
tokenized_corpus = [doc['text'].lower().split() for doc in results]

# Step 2: Create BM25 index
bm25 = BM25Okapi(tokenized_corpus)

# Step 3: Search with BM25
query = "natural language processing toolkit"
tokenized_query = query.lower().split()

# Get BM25 scores for all documents
bm25_scores = bm25.get_scores(tokenized_query)

# Get top k results
top_k = 5
top_indices = np.argsort(bm25_scores)[::-1][:top_k]

# Retrieve top documents
bm25_results = []
for idx in top_indices:
    result = results[idx].copy()
    result['bm25_score'] = float(bm25_scores[idx])
    bm25_results.append(result)

print("BM25 Results:")
for result in bm25_results:
    print(f"Score: {result['bm25_score']:.4f}")
    print(f"Text: {result['text'][:100]}...")
    print()

BM25 Results:
Score: 8.8590
Text: You should now be equipped to work with large datasets, to create robust models of
linguistic phenom...

Score: 1.7753
Text: 80 | Chapter 3:  Processing Raw Text
Notice that NLTK was needed for tokenization, but not for any o...

Score: 1.4878
Text: opening a URL and reading it into a string. If we now take the further step of creating
an NLTK 
tex...

Score: 0.0000
Text: To perform the first three tasks, we can define a function that simply connects together
NLTK’s defa...

Score: 0.0000
Text: saw in Chapter 1, along with the regular list operations, such as slicing:
>>> text = nltk.Text(toke...



In [26]:
bm25_results

[{'chunk_id': '6250bf1e-e76e-4bb9-b33b-d05eb07aa77c',
  'text': 'You should now be equipped to work with large datasets, to create robust models of\nlinguistic phenomena, and to extend them into components for practical language\ntechnologies. We hope that the Natural Language Toolkit (NLTK) has served to open\nup the exciting endeavor of practical natural language processing to a broader audience\nthan before.\nIn spite of all that has come before, language presents us with far more than a temporary\n',
  'metadata': {'filename': 'NLTK.pdf',
   'end_index': 1003848,
   'file_size': 3580937,
   'file_type': 'pdf',
   'chunk_size': 768,
   'chunk_index': 5268,
   'page_number': 463,
   'start_index': 1003412,
   'token_count': 83,
   'content_type': 'application/pdf',
   'embedding_model': 'all-MiniLM-L6-v2',
   'upload_timestamp': '139785087795402850',
   'validation_passed': True,
   'embedding_dimension': 384,
   'similarity_threshold': 0.5},
  'document_id': 'b715603a-d540-4474-87dc

In [34]:
def rerank_bm25(query:str,
                sources: List[Dict], 
                top_k:int):
    """
    Using bm25 to return top k sources from the inputs
    Args:
        sources: 
            - List of sources from postgresql
            - Format: {'chunk_id': '6250bf1e-e76e-4bb9-b33b-d05eb07aa77c',
                        'text': 'abc'
                        'metadata': {},
                        ...}
        top_k: top result
    """
    # Tokenize the text (simple word splitting)
    tokenized_corpus = [doc['text'].lower().split() for doc in sources]

    # Step 2: Create BM25 index
    bm25 = BM25Okapi(tokenized_corpus)

    # Step 3: Search with BM25
    tokenized_query = query.lower().split()

    # Get BM25 scores for all documents
    bm25_scores = bm25.get_scores(tokenized_query)

    # Get top k results
    top_indices = np.argsort(bm25_scores)[::-1][:top_k]

    # Retrieve top documents
    bm25_results = []
    for idx in top_indices:
        result = sources[idx].copy()
        result['bm25_score'] = float(bm25_scores[idx])
        bm25_results.append(result)

    return bm25_results

query = 'What is NLTK?'

test = rerank_bm25(query, results, top_k=2)
len(test)

2

In [35]:
test[0]

{'chunk_id': 'a82fbcec-4ccc-4765-bdaf-bca4fa128d46',
 'text': '... "it means just what I choose it to mean - neither more nor less."\'\'\'\n>>> [w.lower() for w in nltk.word_tokenize(text)]\n[\'"\', \'when\', \'i\', \'use\', \'a\', \'word\', \',\', \'"\', \'humpty\', \'dumpty\', \'said\', ...]\n',
 'metadata': {'filename': 'NLTK.pdf',
  'end_index': 339553,
  'file_size': 3580937,
  'file_type': 'pdf',
  'chunk_size': 768,
  'chunk_index': 1820,
  'page_number': 159,
  'start_index': 339348,
  'token_count': 99,
  'content_type': 'application/pdf',
  'embedding_model': 'all-MiniLM-L6-v2',
  'upload_timestamp': '139785087795402850',
  'validation_passed': True,
  'embedding_dimension': 384,
  'similarity_threshold': 0.5},
 'document_id': 'b715603a-d540-4474-87dc-62f26a56436d',
 'similarity': 0.5444937997452974,
 'bm25_score': 1.9189252625506943}

In [ ]:
# Convert reranked results back to the expected format
results = [
    {
        'chunk_id': r.get('chunk_id'),
        'text': r.get('text'),
        'document_id': r.get('document_id'),
        'metadata': r.get('metadata'),
        'similarity': r.get('similarity'),
        'rerank_score': r.get('bm25_score')
    }
    for r in test
]

avg_rerank_score = sum(r['rerank_score'] for r in results) / len(results) if results else None

AttributeError: 'dict' object has no attribute 'text'

In [1]:
28500000*0.85

24225000.0

# Processs PDF

In [ ]:
import pymupdf
import fitz  # PyMuPDF
import pandas as pd
import os

def parse_and_format_pdf(file_path):
    # Open the document
    doc = fitz.open(file_path)
    print(f"Processing: {os.path.basename(file_path)} | Total Pages: {len(doc)}\n")
    for page_num in range(len(doc)):
        page = doc[page_num]
        print(f"--- [ PAGE {page_num + 1} ] ---")
        # --- 1. HEADERS & SECTIONS (based on Font Size) ---
        # We use .get_text("dict") to get metadata like font size and style
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for l in b["lines"]:
                    for s in l["spans"]:
                        text = s["text"].strip()
                        size = s["size"]
                        # Logic: Larger fonts are typically Headers/Sections
                        if text:
                            if size >= 18:
                                print(f"[Header H1] {text}")
                            elif size >= 14:
                                print(f"[Section H2] {text}")
                            elif size >= 12 and "Bold" in s["font"]:
                                print(f"[Sub-Section] {text}")
        # --- 2. TABLES (PyMuPDF Table Finder) ---
        # Requires PyMuPDF v1.21.0+
        tabs = page.find_tables()
        if tabs.tables:
            print(f"\n[Tables Found: {len(tabs.tables)}]")
            for i, tab in enumerate(tabs.tables):
                # Convert to DataFrame for pretty formatting
                df = tab.to_pandas()
                print(f"Table {i+1} Data:\n{df.to_string(index=False)}\n")
        # --- 3. IMAGES ---
        images = page.get_images(full=True)
        if images:
            print(f"\n[Images Found: {len(images)}]")
            for img_index, img in enumerate(images):
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_ext = base_image["ext"]
                # You can save the image or just report its presence
                # filename = f"page{page_num+1}_img{img_index}.{image_ext}"
                # with open(filename, "wb") as f: f.write(base_image["image"])
                print(f"Image {img_index+1}: Detected (Type: {image_ext.upper()}, Xref: {xref})")
        
        print("-" * 30 + "\n")

In [33]:
import fitz  # PyMuPDF
import pandas as pd

def process_page_to_markdown(doc: fitz.Document, page_num: int) -> str:
    page = doc[page_num]
    
    # 1. IDENTIFY TABLES
    tables = page.find_tables()
    table_bboxes = [tab.bbox for tab in tables]
    
    # 2. EXTRACT BLOCKS
    blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_IMAGES)["blocks"]
    
    items = []
    
    for block in blocks:
        block_bbox = block["bbox"]
        block_rect = fitz.Rect(block_bbox)
        
        # Check table intersection
        is_in_table = False
        for tab_bbox in table_bboxes:
            tab_rect = fitz.Rect(tab_bbox)
            
            # Use & operator for intersection
            intersection = tab_rect & block_rect
            
            # Calculate areas
            block_area = abs(block_rect.x1 - block_rect.x0) * abs(block_rect.y1 - block_rect.y0)
            inter_area = abs(intersection.x1 - intersection.x0) * abs(intersection.y1 - intersection.y0)
            
            if block_area > 0 and (inter_area / block_area) > 0.5:
                is_in_table = True
                break
        
        if is_in_table:
            continue
            
        # [Rest of the logic remains the same...]
        if block["type"] == 0:  # Text
            block_text = ""
            max_size = 0
            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"] + " "
                    if span["size"] > max_size:
                        max_size = span["size"]
            
            block_text = block_text.strip()
            if not block_text: continue

            if max_size > 18: content = f"# {block_text}"
            elif max_size > 14: content = f"## {block_text}"
            else: content = block_text
            
            items.append({"y0": block_rect.y0, "x0": block_rect.x0, "type": "text", "content": content})

        elif block["type"] == 1: # Image
            items.append({"y0": block_rect.y0, "x0": block_rect.x0, "type": "image", "content": "<image> image <image>"})

    # Add Tables
    for table in tables:
        df = table.to_pandas()
        if not df.empty:
            markdown = df.to_markdown(index=False)
            items.append({"y0": table.bbox[1], "x0": table.bbox[0], "type": "table", "content": f"<table>\n{markdown}\n<table>"})

    items.sort(key=lambda x: (x["y0"], x["x0"]))
    return items

In [ ]:
file_path = r"D:\Books\2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf"
doc = fitz.open(file_path)

In [51]:
doc.page_count

510

In [59]:
page_num = 500
items = process_page_to_markdown(doc, page_num)
# Build final string
markdown_output = f"[Page {page_num + 1}]\n\n"
for item in items:
    markdown_output += item["content"] + "\n\n"

output_file = "output_markdown.md"
print(f"Writing output to {output_file}...")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_output)

Writing output to output_markdown.md...


In [49]:
items

[{'y0': 59.25000762939453,
  'x0': 79.44999694824219,
  'type': 'image',
  'content': '<image> image <image>'},
 {'y0': 163.25164794921875,
  'x0': 72.0,
  'type': 'text',
  'content': 'Figure 3-3. Decision threshold and precision/recall tradeoff'},
 {'y0': 187.98814392089844,
  'x0': 71.99626159667969,
  'type': 'text',
  'content': 'Scikit-Learn does not let you set the threshold directly, but it does give you access to the decision scores that it uses to make predictions. Instead of calling the classifier’s predict()  method, you can call its  decision_function()  method, which returns a score for each instance, and then make predictions based on those scores using any threshold you want:'},
 {'y0': 260.1324462890625,
  'x0': 88.9982681274414,
  'type': 'text',
  'content': '>>>  y_scores   =   sgd_clf . decision_function ([ some_digit ]) >>>  y_scores array([2412.53175101]) >>>  threshold   =   0 >>>  y_some_digit_pred   =  ( y_scores   >   threshold ) array([ True])'},
 {'y0': 325

In [38]:
markdown_output

'[Page 121]\n\ntreats all values equally, the harmonic mean gives much more weight to low values. As a result, the classifier will only get a high F 1  score if both recall and precision are high.\n\nEquation 3-3. F 1\n\nF 1  = 2 1 precision   + 1 recall = 2 ×   precision × recall precision + recall   = TP\n\nTP  +   FN  +  FP 2\n\nTo compute the F 1  score, simply call the  f1_score()  function:\n\n>>>  from   sklearn.metrics   import   f1_score >>>  f1_score ( y_train_5 ,  y_train_pred ) 0.7420962043663375\n\nThe F 1  score favors classifiers that have similar precision and recall. This is not always what you want: in some contexts you mostly care about precision, and in other con‐ texts you really care about recall. For example, if you trained a classifier to detect vid‐ eos that are safe for kids, you would probably prefer a classifier that rejects many good videos (low recall) but keeps only safe ones (high precision), rather than a clas‐ sifier that has a much higher recall but l

In [39]:
# for page_num in range(len(doc)):
#     page = doc[page_num]
#     print(f"--- [ PAGE {page_num + 1} ] ---")
#     # --- 1. HEADERS & SECTIONS (based on Font Size) ---
#     # We use .get_text("dict") to get metadata like font size and style
#     blocks = page.get_text("dict")["blocks"]
#     for b in blocks:
#         if "lines" in b:
#             for l in b["lines"]:
#                 for s in l["spans"]:
#                     text = s["text"].strip()
#                     size = s["size"]
#                     # Logic: Larger fonts are typically Headers/Sections
#                     if text:
#                         if size >= 18:
#                             print(f"[Header H1] {text}")
#                         elif size >= 14:
#                             print(f"[Section H2] {text}")
#                         elif size >= 12 and "Bold" in s["font"]:
#                             print(f"[Sub-Section] {text}")

In [ ]:
# for page_num in range(len(doc)):
#     page = doc[page_num]
#     tabs = page.find_tables()
#     if tabs.tables:
#         print(f"\n[Tables Found: {len(tabs.tables)}] in {page_num}")
#         for i, tab in enumerate(tabs.tables):
#             # Convert to DataFrame for pretty formatting
#             df = tab.to_pandas()
#             print(f"Table {i+1} Data:\n{df.to_string(index=False)}\n")

page = doc[page_num]

# 1. IDENTIFY TABLES
# ------------------
# We find tables first to get their bounding boxes.
# This helps us avoid duplicating text that is already inside a table.
tables = page.find_tables()
table_bboxes = [tab.bbox for tab in tables]

# 2. EXTRACT ALL BLOCKS (Text & Images)
# -------------------------------------
# get_text("dict") provides detailed layout info.
# flags=fitz.TEXT_PRESERVE_IMAGES ensures image blocks are included.
blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_IMAGES)["blocks"]
blocks

[{'type': 0,
  'number': 0,
  'flags': 0,
  'bbox': (71.99696350097656,
   52.29900360107422,
   432.0000915527344,
   91.66349792480469),
  'lines': [{'spans': [{'size': 10.5,
      'flags': 4,
      'bidi': 0,
      'char_flags': 16,
      'font': 'MinionPro-Regular',
      'color': 0,
      'alpha': 255,
      'ascender': 0.9890000224113464,
      'descender': -0.36000001430511475,
      'text': 'treats all values equally, the harmonic mean gives much more weight to low values.',
      'origin': (72.0, 62.683502197265625),
      'bbox': (72.0, 52.29900360107422, 432.0000915527344, 66.4635009765625)}],
    'wmode': 0,
    'dir': (1.0, 0.0),
    'bbox': (72.0, 52.29900360107422, 432.0000915527344, 66.4635009765625)},
   {'spans': [{'size': 10.5,
      'flags': 4,
      'bidi': 0,
      'char_flags': 16,
      'font': 'MinionPro-Regular',
      'color': 0,
      'alpha': 255,
      'ascender': 0.9890000224113464,
      'descender': -0.36000001430511475,
      'text': 'As a result, the 

In [47]:
blocks[0]['lines'][0]['spans']

[{'size': 10.5,
  'flags': 4,
  'bidi': 0,
  'char_flags': 16,
  'font': 'MinionPro-Regular',
  'color': 0,
  'alpha': 255,
  'ascender': 0.9890000224113464,
  'descender': -0.36000001430511475,
  'text': 'treats all values equally, the harmonic mean gives much more weight to low values.',
  'origin': (72.0, 62.683502197265625),
  'bbox': (72.0, 52.29900360107422, 432.0000915527344, 66.4635009765625)}]

# Test with Gemini Models

In [1]:
import io
import os
import re
from typing import Optional

import fitz
from dotenv import load_dotenv
from PIL import Image

load_dotenv()

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
TEXT_DENSITY_THRESHOLD = 100       # min chars for fitz to handle the page
IMAGE_AREA_RATIO_THRESHOLD = 0.6   # if images cover >60% of page → VLM
VLM_DPI = 200                      # render DPI when sending page to VLM
IMAGE_CROP_DPI = 150               # DPI for cropped image/table regions

# Table quality thresholds for rescue logic
TABLE_MIN_ROWS = 2                 # tables with fewer rows are "suspicious"
TABLE_EMPTY_CELL_RATIO = 0.5      # tables with >50% empty cells are "suspicious"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# ---------------------------------------------------------------------------
# Gemini client (lazy singleton)
# ---------------------------------------------------------------------------

_gemini_model = None


def _get_gemini_model():
    """Initialise and cache the Gemini GenerativeModel."""
    global _gemini_model
    if _gemini_model is not None:
        return _gemini_model

    try:
        import google.generativeai as genai
    except ImportError as exc:
        raise ImportError(
            "google-generativeai is required. Install with: "
            "pip install google-generativeai"
        ) from exc

    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError(
            "GOOGLE_API_KEY environment variable is not set. "
            "Add it to your .env file."
        )

    genai.configure(api_key=api_key)
    _gemini_model = genai.GenerativeModel(GEMINI_MODEL)
    return _gemini_model


def _png_bytes_to_pil(png_bytes: bytes) -> Image.Image:
    return Image.open(io.BytesIO(png_bytes))


# ---------------------------------------------------------------------------
# VLM prompts
# ---------------------------------------------------------------------------

VLM_PAGE_PROMPT = """\
Convert this PDF page to clean markdown.
- Use # / ## / ### for headings based on visual size.
- Render tables as GitHub-flavoured markdown tables.
- For charts or diagrams write a short description inside an <image> tag.
- Preserve the reading order of the original page.
- Output only the markdown — no commentary or preamble.
"""

VLM_IMAGE_PROMPT = """\
Describe this image concisely for embedding in a markdown document.
- If it is a table, render it as a GitHub-flavoured markdown table.
- If it is a chart or diagram, describe what it shows in 1–3 sentences.
- If it is a photograph or illustration, describe its content briefly.
- Output only the description or markdown table — no commentary.
"""

VLM_TABLE_RESCUE_PROMPT = """\
This is a cropped region from a PDF page that contains a table.
Render the table as a GitHub-flavoured markdown table.
Preserve all cell values exactly as they appear.
Output only the markdown table — no commentary.
"""


# ---------------------------------------------------------------------------
# Gemini API calls
# ---------------------------------------------------------------------------

def call_vlm(png_bytes: bytes, prompt: str) -> str:
    """
    Send a full page image to Gemini and return markdown.

    Args:
        png_bytes: PNG-encoded page image.
        prompt: Instruction prompt for the model.

    Returns:
        Markdown string produced by Gemini.
    """
    try:
        model = _get_gemini_model()
        img = _png_bytes_to_pil(png_bytes)
        response = model.generate_content([img, prompt])
        return response.text
    except Exception as exc:
        print(f"  [VLM page] Gemini call failed: {exc}")
        return f"<!-- VLM page extraction failed: {exc} -->"


def call_vlm_for_region(png_bytes: bytes, prompt: str = VLM_IMAGE_PROMPT) -> str:
    """
    Send a cropped image/table region to Gemini for description or transcription.

    Args:
        png_bytes: PNG-encoded region image.
        prompt: Instruction prompt (defaults to VLM_IMAGE_PROMPT).

    Returns:
        Markdown description or table produced by Gemini.
    """
    try:
        model = _get_gemini_model()
        img = _png_bytes_to_pil(png_bytes)
        response = model.generate_content([img, prompt])
        return response.text
    except Exception as exc:
        print(f"  [VLM region] Gemini call failed: {exc}")
        return f"<!-- VLM region extraction failed: {exc} -->"


# ---------------------------------------------------------------------------
# Page classifier
# ---------------------------------------------------------------------------

def classify_page(page: fitz.Page) -> str:
    """
    Return 'fitz' or 'vlm' based on text density and image coverage.

    A page is sent to the VLM when:
    - It has too little extractable text (likely scanned), OR
    - Images cover more than IMAGE_AREA_RATIO_THRESHOLD of the page area.
    """
    text = page.get_text().strip()
    if len(text) < TEXT_DENSITY_THRESHOLD:
        return "vlm"

    image_blocks = [
        b for b in page.get_text("dict", flags=fitz.TEXT_PRESERVE_IMAGES)["blocks"]
        if b["type"] == 1
    ]
    image_area = sum(
        (b["bbox"][2] - b["bbox"][0]) * (b["bbox"][3] - b["bbox"][1])
        for b in image_blocks
    )
    page_area = page.rect.width * page.rect.height

    if page_area > 0 and (image_area / page_area) > IMAGE_AREA_RATIO_THRESHOLD:
        return "vlm"

    return "fitz"


# ---------------------------------------------------------------------------
# Image description helpers (used on fitz path)
# ---------------------------------------------------------------------------

def _crop_block(page: fitz.Page, bbox: tuple, dpi: int = IMAGE_CROP_DPI) -> bytes:
    """Render a bounding-box region of a page to PNG bytes."""
    rect = fitz.Rect(bbox)
    # Add a small padding so we don't clip edge pixels
    padded = rect + fitz.Rect(-2, -2, 2, 2)
    clip = padded & page.rect          # clamp to page bounds
    pix = page.get_pixmap(clip=clip, dpi=dpi)
    return pix.tobytes("png")


def describe_page_images(page: fitz.Page) -> list[str]:
    """
    For every image block on a page, crop it and ask Gemini to describe it.

    Returns a list of descriptions in top-to-bottom (y0) order, matching the
    order in which '<image> image <image>' placeholders appear in the fitz
    markdown output.
    """
    blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_IMAGES)["blocks"]
    image_blocks = sorted(
        (b for b in blocks if b["type"] == 1),
        key=lambda b: b["bbox"][1],
    )

    descriptions = []
    for block in image_blocks:
        png_bytes = _crop_block(page, block["bbox"])
        desc = call_vlm_for_region(png_bytes, VLM_IMAGE_PROMPT)
        descriptions.append(desc.strip())

    return descriptions


def _replace_image_placeholders(markdown: str, descriptions: list[str]) -> str:
    """
    Replace each '<image> image <image>' placeholder in `markdown` with the
    corresponding Gemini description (in order of appearance).
    """
    placeholder = "<image> image <image>"
    for desc in descriptions:
        replacement = f"<image>\n\n{desc}\n\n</image>"
        markdown = markdown.replace(placeholder, replacement, 1)
    return markdown


# ---------------------------------------------------------------------------
# fitz path
# ---------------------------------------------------------------------------

def process_page_fitz(
    doc: fitz.Document,
    page: fitz.Page,
    page_num: int,
    last_table_headers: Optional[list],
    describe_images: bool = True,
) -> tuple:
    """
    Run the fitz-based extraction, then optionally send every inline image
    to Gemini for a richer description.

    Returns:
        (markdown_str, updated_last_table_headers)
    """
    from process_pdf_to_md import process_page_to_markdown

    page_md, updated_headers = process_page_to_markdown(doc, page_num, last_table_headers)

    if describe_images and "<image> image <image>" in page_md:
        descriptions = describe_page_images(page)
        if descriptions:
            page_md = _replace_image_placeholders(page_md, descriptions)

    return page_md, updated_headers


# ---------------------------------------------------------------------------
# VLM path
# ---------------------------------------------------------------------------

def process_page_vlm(page: fitz.Page, page_num: int) -> str:
    """
    Render the entire page to PNG and send to Gemini for markdown conversion.

    Used for scanned or image-dominated pages where fitz text extraction
    would yield poor results.
    """
    pix = page.get_pixmap(dpi=VLM_DPI)
    png_bytes = pix.tobytes("png")
    vlm_output = call_vlm(png_bytes, VLM_PAGE_PROMPT)
    return f"[Page {page_num + 1}]\n\n{vlm_output}\n\n"


# ---------------------------------------------------------------------------
# Table rescue (fitz path quality check)
# ---------------------------------------------------------------------------

def _is_suspicious_table(table_md: str) -> bool:
    """
    Return True if a markdown table looks low-quality:
    - Fewer than TABLE_MIN_ROWS data rows, OR
    - More than TABLE_EMPTY_CELL_RATIO fraction of cells are empty.
    """
    lines = [l for l in table_md.splitlines() if l.strip().startswith("|")]
    # Remove separator row (---|---)
    data_lines = [l for l in lines if not re.match(r"^\s*\|[\s\-|:]+\|\s*$", l)]

    if len(data_lines) < TABLE_MIN_ROWS:
        return True

    total_cells = 0
    empty_cells = 0
    for line in data_lines:
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        total_cells += len(cells)
        empty_cells += sum(1 for c in cells if not c)

    if total_cells > 0 and (empty_cells / total_cells) > TABLE_EMPTY_CELL_RATIO:
        return True

    return False


def maybe_rescue_tables(page: fitz.Page, markdown: str) -> str:
    """
    Find <table> blocks in `markdown` that look low-quality and re-extract
    them by sending the corresponding page region to Gemini.

    Matching works by sequential index: the N-th <table> block in the markdown
    corresponds to the N-th table detected by page.find_tables().
    """
    # Find all <table>…<table> blocks
    table_pattern = re.compile(r"<table>\n(.*?)\n<table>", re.DOTALL)
    table_matches = list(table_pattern.finditer(markdown))

    if not table_matches:
        return markdown

    detected_tables = page.find_tables().tables  # fitz TableFinder result

    result = markdown
    offset = 0  # track string position shift after replacements

    for idx, match in enumerate(table_matches):
        table_md = match.group(1)

        if not _is_suspicious_table(table_md):
            continue

        if idx >= len(detected_tables):
            break  # no matching fitz table to crop

        fitz_table = detected_tables[idx]
        png_bytes = _crop_block(page, fitz_table.bbox)
        rescued_md = call_vlm_for_region(png_bytes, VLM_TABLE_RESCUE_PROMPT).strip()

        old_block = match.group(0)
        new_block = f"<table>\n{rescued_md}\n<table>"

        # Adjust for previous replacements shifting positions
        start = match.start() + offset
        end = match.end() + offset
        result = result[:start] + new_block + result[end:]
        offset += len(new_block) - len(old_block)

    return result


In [16]:

# ---------------------------------------------------------------------------
# Main hybrid processor
# ---------------------------------------------------------------------------

def process_pdf_hybrid(
    doc: fitz.Document,
    describe_images: bool = True,
    rescue_tables: bool = True,
) -> str:
    """
    Process a PDF document page-by-page using the hybrid fitz + Gemini strategy.

    Args:
        doc: Open fitz Document.
        describe_images: If True, inline images on fitz-processed pages are
            described by Gemini (replacing the '<image>' placeholders).
        rescue_tables: If True, low-quality tables on fitz-processed pages are
            re-extracted by Gemini.

    Returns:
        Full document markdown as a single string.
    """
    full_markdown = ""
    last_table_headers: Optional[list] = None
    total = len(doc)

    for page_num in range(30,40):
        page = doc[page_num]
        route = classify_page(page)

        try:
            if route == "fitz":
                page_md, last_table_headers = process_page_fitz(
                    doc, page, page_num, last_table_headers,
                    describe_images=describe_images,
                )
                if rescue_tables:
                    page_md = maybe_rescue_tables(page, page_md)
            else:
                page_md = process_page_vlm(page, page_num)
                last_table_headers = None  # VLM handles continuity implicitly

            full_markdown += page_md + "\n\n---\n\n"

        except Exception as exc:
            print(f"[page {page_num + 1}] error ({route}): {exc}")
            last_table_headers = None

        if (page_num + 1) % 10 == 0:
            print(f"  {page_num + 1}/{total} pages processed...")

    return full_markdown

In [17]:
file_path = r"D:\Books\2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf"
output_file = "output_markdown_test.md"

if os.path.exists(file_path):
    doc = fitz.open(file_path)

md = process_pdf_hybrid(
    doc,
    describe_images=True,
    rescue_tables=True,
)
doc.close()

with open(output_file, "w", encoding="utf-8") as f:
    f.write(md)
print(f"Saved → {output_file}")

  [VLM region] Gemini call failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 14.583133668s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 14
}
]
  [VLM region] Gemini call failed: 429 You exceeded your current quota, p

In [7]:
if os.path.exists(file_path):
    doc = fitz.open(file_path)

In [12]:
doc = fitz.open(file_path)

FileNotFoundError: no such file: 'D:/Books/2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O'Reilly-Media-2019.pdf'

In [1]:
from enum import Enum
import re
import os
from chonkie import TokenChunker, RecursiveChunker, SemanticChunker

def read_markdown_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()

# Example usage:
markdown_content = read_markdown_file(r"output_markdown_test.md")

class ChunkerType(Enum):
    """Available chunking strategies."""
    TOKEN = "token"
    RECURSIVE = "recursive"
    MARKDOWN = "markdown"
    SEMANTIC = "semantic"


# Size threshold for adaptive chunking (100KB)
LARGE_DOCUMENT_THRESHOLD_CHARS = 100_000

# Global cache for chunkers (especially important for SemanticChunker)
_CHUNKER_CACHE = {}


def get_chunker(
    chunker_type: Optional[str] = None,
    chunk_size: int = 512,
    chunk_overlap: int = 50,
    similarity_threshold: float = 0.5,
    embedding_model: Optional[str] = None,
    text_length: Optional[int] = None
):
    """
    Factory function to create the appropriate chunker.

    The expected input text is **Markdown** (produced by PDFToMarkdownConverter).
    Choose a chunker strategy that best fits your quality / speed trade-off.

    Args:
        chunker_type: Type of chunker ("token", "recursive", "markdown", "semantic")
                     If None, uses CHUNKER_TYPE env var or defaults to "markdown"
        chunk_size: Maximum tokens per chunk
        chunk_overlap: Number of overlapping tokens/sentences
        similarity_threshold: For semantic chunker only (0-1)
        embedding_model: For semantic chunker only
        text_length: Optional text length for adaptive chunker selection.
                    If provided and > LARGE_DOCUMENT_THRESHOLD_CHARS,
                    forces RecursiveChunker for performance.

    Returns:
        Configured chunker instance

    Examples:
        >>> # Use default (markdown-aware)
        >>> chunker = get_chunker(chunk_size=512)

        >>> # Use markdown chunker explicitly
        >>> chunker = get_chunker("markdown", chunk_size=512)

        >>> # Use token chunker explicitly
        >>> chunker = get_chunker("token", chunk_size=512)

        >>> # Use semantic chunker for small documents
        >>> chunker = get_chunker("semantic", chunk_size=512, similarity_threshold=0.3)

        >>> # Adaptive selection based on document size
        >>> chunker = get_chunker("semantic", text_length=500000)  # Will use recursive
    """
    

    # Determine chunker type from env or parameter
    if chunker_type is None:
        chunker_type = os.getenv("CHUNKER_TYPE", "markdown").lower()
    else:
        chunker_type = chunker_type.lower()

    # Validate parameters
    if chunk_size < chunk_overlap:
        error_msg = f"Chunk size ({chunk_size}) must be greater than chunk overlap ({chunk_overlap})"
        logfire.error(error_msg, chunk_size=chunk_size,
                      chunk_overlap=chunk_overlap)
        raise ValueError(error_msg)

    # Adaptive selection: Force RecursiveChunker for large documents
    # This prevents SemanticChunker from being too slow on large markdown documents
    if text_length is not None and text_length > LARGE_DOCUMENT_THRESHOLD_CHARS:
        if chunker_type == ChunkerType.SEMANTIC.value:
            print(f"[Adaptive] Document is large ({text_length:,} chars > {LARGE_DOCUMENT_THRESHOLD_CHARS:,}). "
                  f"Switching from SemanticChunker to RecursiveChunker for performance.")
            chunker_type = ChunkerType.RECURSIVE.value

    # Create appropriate chunker
    if chunker_type == ChunkerType.TOKEN.value:
        cache_key = f"token_{chunk_size}_{chunk_overlap}"
        if cache_key not in _CHUNKER_CACHE:
            _CHUNKER_CACHE[cache_key] = TokenChunker(
                tokenizer="character",
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            )
        return _CHUNKER_CACHE[cache_key]

    elif chunker_type == ChunkerType.RECURSIVE.value:
        # Uses hierarchical text splitting with generic delimiters
        cache_key = f"recursive_{chunk_size}"
        if cache_key not in _CHUNKER_CACHE:
            _CHUNKER_CACHE[cache_key] = RecursiveChunker(
                tokenizer="character",
                chunk_size=chunk_size
            )
        return _CHUNKER_CACHE[cache_key]

    elif chunker_type == ChunkerType.MARKDOWN.value:
        # Markdown-aware chunker: uses chonkie's built-in markdown recipe
        # which splits on headings, lists, code blocks, etc.
        # This is the recommended chunker when input is markdown from PDFToMarkdownConverter.
        cache_key = f"markdown_{chunk_size}"
        if cache_key not in _CHUNKER_CACHE:
            _CHUNKER_CACHE[cache_key] = RecursiveChunker.from_recipe(
                "markdown",
                lang="en",
                chunk_size=chunk_size
            )
        return _CHUNKER_CACHE[cache_key]

    elif chunker_type == ChunkerType.SEMANTIC.value:
        # Semantic chunker is HEAVY (loads models). Must cache.
        embedding_model_name = embedding_model or "sentence-transformers/all-MiniLM-L6-v2"
        cache_key = f"semantic_{chunk_size}_{similarity_threshold}_{embedding_model_name}"

        if cache_key not in _CHUNKER_CACHE:
            print(
                f"Initializing SemanticChunker (loading model: {embedding_model_name})...")
            _CHUNKER_CACHE[cache_key] = SemanticChunker(
                chunk_size=chunk_size,
                threshold=similarity_threshold,
                embedding_model=embedding_model_name
            )
        return _CHUNKER_CACHE[cache_key]

    else:
        raise ValueError(
            f"Invalid chunker_type: {chunker_type}. "
            f"Must be one of: {[e.value for e in ChunkerType]}"
        )

def chunk_markdown(
    markdown_text: str,
    chunker_type: Optional[str] = None,
    chunk_size: int = 512,
    chunk_overlap: int = 50,
    similarity_threshold: float = 0.5,
    embedding_model: Optional[str] = None
) -> List:
    """
    Convenience function to chunk markdown text with the specified strategy.

    Args:
        markdown_text: Markdown content to chunk (from PDFToMarkdownConverter)
        chunker_type: Type of chunker (None = use default/env var, defaults to "markdown")
        chunk_size: Maximum tokens per chunk
        chunk_overlap: Overlap size
        similarity_threshold: For semantic chunker
        embedding_model: For semantic chunker

    Returns:
        List of chunks

    Examples:
        >>> chunks = chunk_markdown(md_text)  # Uses default (markdown-aware)
        >>> chunks = chunk_markdown(md_text, chunker_type="token")  # Fast
        >>> chunks = chunk_markdown(md_text, chunker_type="semantic")  # Quality
    """
    chunker = get_chunker(
        chunker_type=chunker_type,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        similarity_threshold=similarity_threshold,
        embedding_model=embedding_model
    )

    chunks = chunker.chunk(markdown_text)

    page_markers = list(re.finditer(r'\[Page (\d+)\]', markdown_text))
    
    if page_markers:
        # Map each page number → its content slice
        page_content_map: dict = {}
        for i, marker in enumerate(page_markers):
            page_num = int(marker.group(1))
            start = marker.end()
            end = page_markers[i + 1].start() if i + 1 < len(page_markers) else len(markdown_text)
            page_content_map[page_num] = markdown_text[start:end].strip()

        # Assign each chunk only the content of its own page (not the full document)
        first_page = int(page_markers[0].group(1))
        for chunk in chunks:
            start_idx = getattr(chunk, 'start_index', None)
            if start_idx is None:
                start_idx = 0
            chunk_page = first_page
            for marker in page_markers:
                if marker.start() > start_idx:
                    break
                chunk_page = int(marker.group(1))
            setattr(chunk, 'full_content', page_content_map.get(chunk_page, ''))
            setattr(chunk, 'page', chunk_page)
    else:
        # No page markers (plain markdown / non-PDF): store the full text as fallback
        for chunk in chunks:
            setattr(chunk, 'full_content', markdown_text)
            setattr(chunk, 'page', 'all')

    return chunks

In [2]:
chunks = chunk_markdown(markdown_content, chunker_type="token")  # Fast

# Docling

In [ ]:
# from docling.document_converter import DocumentConverter

# source = r"D:\Books\2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf"
# converter = DocumentConverter()
# doc = converter.convert(source).document
# print(doc.export_to_markdown())

In [12]:
import io, os
import time
from pathlib import Path
from dotenv import load_dotenv
from PIL import Image
import google.generativeai as genai

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling_core.types.doc import (
    TextItem, SectionHeaderItem, TableItem, PictureItem, ListItem
)

load_dotenv()

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
GEMINI_IMAGE_PROMPT = """\
Describe this image concisely for embedding in a markdown document.
- If it is a chart or diagram, describe what it shows in 1-3 sentences.
- If it is a photograph or illustration, describe its content briefly.
Output only the description — no commentary.
"""

_gemini = None
def get_gemini():
    global _gemini
    if _gemini is None:
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        _gemini = genai.GenerativeModel(GEMINI_MODEL)
    return _gemini

def describe_image_gemini(pil_image: Image.Image, sleep_seconds: float=2.0) -> str:
    try:
        response = get_gemini().generate_content([pil_image, GEMINI_IMAGE_PROMPT]).text.strip()
        time.sleep(sleep_seconds)
        return response
        
    except Exception as e:
        return f"<!-- Image description failed: {e} -->"


In [18]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from pathlib import Path

# Fix Windows WinError 1314: download models to local_dir (uses copy, not symlinks)
DOCLING_MODELS_PATH = Path.home() / "docling_models"
DOCLING_MODELS_PATH.mkdir(exist_ok=True)

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.artifacts_path = DOCLING_MODELS_PATH   # ← key fix

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)


In [14]:
def pdf_to_markdown_docling(pdf_path: str, num_pages: int, describe_images: bool = True) -> str:
    """
    Docling (structure + tables) + Gemini (images only).
    Output uses [Page N] markers → compatible with chunk_markdown().
    """
    result = converter.convert(pdf_path)
    doc = result.document

    # Group elements by page, keeping their vertical position for sorting
    pages: dict[int, list] = {}
    for item, _ in doc.iterate_items():
        if not getattr(item, 'prov', None):
            continue
        prov = item.prov[0]
        page_no = prov.page_no
        # bbox.t = top edge; sort ascending = top-to-bottom reading order
        # (docling normalises to top-left origin)
        bbox_t = prov.bbox.t if prov.bbox else 0
        pages.setdefault(page_no, []).append((bbox_t, item))

    pages = {k: v for k, v in pages.items() if k < num_pages} # test on 20 pages 
    
    # Sort each page top-to-bottom
    for page_no in pages:
        pages[page_no].sort(key=lambda x: x[0])

    page_blocks = []
    for page_no in sorted(pages.keys()):
        lines = [f"[Page {page_no}]"]

        for _, item in pages[page_no]:

            if isinstance(item, SectionHeaderItem):
                level = getattr(item, 'level', 1)
                lines.append(f"{'#' * level} {item.text}")

            elif isinstance(item, TableItem):
                lines.append(item.export_to_markdown())

            elif isinstance(item, PictureItem):
                if describe_images:
                    try:
                        pil_img = item.get_image(doc)
                        desc = describe_image_gemini(pil_img, sleep_seconds=2.0)
                        lines.append(f"<image>\n\n{desc}\n\n</image>")
                    except Exception as e:
                        lines.append(f"<!-- Image extraction failed: {e} -->")
                # skip figures entirely if describe_images=False

            elif isinstance(item, ListItem):
                marker = f"{item.enumerated}." if getattr(item, 'enumerated', False) else "-"
                lines.append(f"{marker} {item.text}")

            elif isinstance(item, TextItem):
                lines.append(item.text)

        page_blocks.append("\n\n".join(lines))

    return "\n\n---\n\n".join(page_blocks)


In [19]:
PDF_PATH = r"D:\Books\2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf"
markdown_with_images = pdf_to_markdown_docling(PDF_PATH, num_pages=20, describe_images=True)
print(markdown_with_images[:3000])


FileNotFoundError: Missing safe tensors file: C:\Users\vietl\docling_models\model.safetensors

In [1]:
import re
import os
import time
import logging
from collections import defaultdict, deque
from pathlib import Path
from typing import Optional

from PIL import Image as PILImage

# ── Docling ───────────────────────────────────────────────────────────────────
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import (
    TableItem,
    PictureItem,
    TextItem,
    SectionHeaderItem,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("hybrid_parser")

# ── Prompts ───────────────────────────────────────────────────────────────────
VLM_TABLE_PROMPT = """\
This is a cropped region from a PDF page containing a table.
Extract and render the table content.

Rules:
- Wrap the output in <table></table> tags (closing tag must be </table>).
- Optionally include <table_caption>Title</table_caption> before the table \
if a caption or title is visible above or below it.
- Use GitHub-flavoured markdown table syntax.
- Separator row must use ONLY dashes: |-|-| (no colons, no padding spaces).
- Do NOT use any HTML inside cells (no <br>, no <td>, no <tr>).
- If a cell contains multiple lines of text, join them with a single space \
in the SAME cell.
- Ensure every row has the same number of columns.
- If a cell is merged vertically (one value applies to multiple rows), keep \
the value in the top row only.
- If the table has multiple header rows, include all of them in order.
- Preserve all cell values exactly as they appear.
- Output only the <table>...</table> block — no commentary or preamble.
- Do NOT use code fences.
"""

VLM_IMAGE_PROMPT = """\
This is an image, chart, diagram, figure, or visual element cropped from a PDF page.
Extract and preserve ALL content inside <figure></figure> tags.

Rules:
- Transcribe ALL visible text EXACTLY as it appears, in its original language.
  Do NOT translate, paraphrase, or describe text — copy it verbatim.
- If the image contains Japanese text, output it in Japanese. NEVER translate it to English.
- Do NOT write any English description or summary of the diagram structure.
- Preserve logical reading order (top-to-bottom, left-to-right within each column).
- For diagrams or flowcharts: transcribe ONLY the visible text labels, captions,
  and annotations verbatim in the original language. No English descriptions.
- For charts/graphs: transcribe all axis labels, legend entries, and data values.
- For colored boxes, banners, or callout regions: include ALL text inside them.
- If the image also contains a table, render it as a GFM markdown table.
- ALL transcribed text must be placed inside the <figure>...</figure> block.
- Output only the <figure>...</figure> block — no commentary or preamble.
- Do NOT use code fences.
"""

# ── Config ────────────────────────────────────────────────────────────────────
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = "gemini-2.5-flash-lite"
GEMINI_RPM_LIMIT = 5    # safe limit (model allows 30, keep buffer)
IMAGES_SCALE = 1.0   # docling render resolution (2× = ~144 dpi equivalent)
COMPLEX_TABLE_ROWS = 8     # tables with more rows  → Gemini
COMPLEX_TABLE_COLS = 6     # tables with more cols  → Gemini


# ── Rate limiter ──────────────────────────────────────────────────────────────
class RateLimiter:
    def __init__(self, max_calls: int = GEMINI_RPM_LIMIT, period: float = 60.0):
        self.max_calls = max_calls
        self.period = period
        self._calls: deque = deque()

    def wait(self):
        while True:
            now = time.monotonic()
            while self._calls and now - self._calls[0] >= self.period:
                self._calls.popleft()
            if len(self._calls) < self.max_calls:
                break
            sleep_for = self.period - (now - self._calls[0])
            logger.info(
                f"[rate limiter] quota reached, sleeping {sleep_for:.1f}s …")
            time.sleep(sleep_for)
        self._calls.append(time.monotonic())


# ── Post-processing helpers ───────────────────────────────────────────────────
def _normalize_table(table_md: str) -> str:
    lines = table_md.splitlines()
    result: list[str] = []
    for line in lines:
        if not line.strip().startswith("|"):
            result.append(line)
            continue
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        if all(re.fullmatch(r"[-:\s]+", c) for c in cells if c):
            result.append("|" + "|".join(["-"] * len(cells)) + "|")
            continue
        if cells and cells[0] == "" and any(c for c in cells[1:]):
            if result:
                prev = [c.strip()
                        for c in result[-1].strip().strip("|").split("|")]
                for i, cont in enumerate(cells):
                    if cont and i < len(prev):
                        prev[i] = (prev[i] + " " + cont).strip()
                result[-1] = "| " + " | ".join(prev) + " |"
                continue
        result.append("| " + " | ".join(cells) + " |")
    return "\n".join(result)


def _clean_html(md: str) -> str:
    md = re.sub(r"\s*<br\s*/?>\s*", " ", md, flags=re.IGNORECASE)
    md = re.sub(r"</?t[rdh]\b[^>]*>",  "", md, flags=re.IGNORECASE)
    return md


def _strip_code_fences(md: str) -> str:
    md = re.sub(r"^```(?:markdown)?\s*\n?", "", md, flags=re.IGNORECASE)
    md = re.sub(r"\n?```\s*$",             "", md, flags=re.IGNORECASE)
    return md.strip()


def _fix_table_closing_tags(md: str) -> str:
    lines, result, in_table = md.splitlines(), [], False
    for line in lines:
        s = line.strip()
        if s == "<table>":
            if in_table:
                result.append("</table>")
                in_table = False
            else:
                result.append(line)
                in_table = True
        elif s == "</table>":
            result.append(line)
            in_table = False
        else:
            result.append(line)
    return "\n".join(result)


def normalize_tables_in_markdown(md: str) -> str:
    out, buf = [], []

    def flush():
        if buf:
            out.extend(_normalize_table("\n".join(buf)).splitlines())
            buf.clear()
    for line in md.splitlines():
        if line.strip().startswith("|"):
            buf.append(line)
        else:
            flush()
            out.append(line)
    flush()
    return "\n".join(out)


# ── Main parser ───────────────────────────────────────────────────────────────
class HybridDoclingGeminiParser:
    """
    Hybrid PDF → Markdown parser using only Docling + Gemini (no fitz).

    Image acquisition (all via docling):
      PictureItem  → item.get_image(doc)            (pre-cropped by docling)
      TableItem    → item.get_image(doc)            (cropped from page image)
      Page image   → doc.pages[n].image.pil_image   (full page, for reference)

    Docling pipeline options that make this work:
      generate_page_images=True       required for item.get_image() fallback
      generate_picture_images=True    pre-crops PictureItems
      images_scale=IMAGES_SCALE       controls resolution of all rendered images
    """

    def __init__(
        self,
        api_key: str = GOOGLE_API_KEY,
        gemini_model: str = GEMINI_MODEL,
        rpm_limit: int = GEMINI_RPM_LIMIT,
        images_scale: float = IMAGES_SCALE,
        complex_table_rows: int = COMPLEX_TABLE_ROWS,
        complex_table_cols: int = COMPLEX_TABLE_COLS,
    ):
        self._api_key = api_key
        self._gemini_model = gemini_model
        self._rate_limiter = RateLimiter(rpm_limit)
        self._images_scale = images_scale
        self._complex_table_rows = complex_table_rows
        self._complex_table_cols = complex_table_cols
        self._genai_model = None
        self._vlm_calls: int = 0

    # ── Docling converter ─────────────────────────────────────────────────────

    def _build_converter(self) -> DocumentConverter:
        opts = PdfPipelineOptions()
        opts.do_ocr = False
        opts.do_table_structure = True
        opts.table_structure_options = TableStructureOptions(
            do_cell_matching=True)
        opts.accelerator_options = AcceleratorOptions(
            num_threads=4, device=AcceleratorDevice.AUTO
        )
        # ── Image rendering (replaces fitz) ──────────────────────────────
        opts.generate_page_images = True   # enables item.get_image(doc) crop
        opts.generate_picture_images = True   # pre-crops PictureItems
        opts.images_scale = self._images_scale
        return DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(
                pipeline_options=opts)}
        )

    # ── Gemini client ─────────────────────────────────────────────────────────

    def _get_model(self):
        if self._genai_model is not None:
            return self._genai_model
        import google.generativeai as genai
        genai.configure(api_key=self._api_key)
        self._genai_model = genai.GenerativeModel(self._gemini_model)
        return self._genai_model

    def _call_gemini(self, pil_img: PILImage.Image, prompt: str, retries: int = 3) -> str:
        """Rate-limited Gemini call. Accepts a PIL Image directly — no bytes conversion."""
        self._rate_limiter.wait()
        model = self._get_model()
        for attempt in range(retries):
            try:
                self._vlm_calls += 1
                return model.generate_content([pil_img, prompt]).text
            except Exception as exc:
                err = str(exc).lower()
                if "429" in err or "quota" in err or "resource_exhausted" in err:
                    if "per_day" in err or "day" in err:
                        raise RuntimeError(
                            "[Gemini] Daily quota exhausted.") from exc
                    wait = 60
                    logger.warning(
                        f"RPM limit hit (attempt {attempt+1}/{retries}), waiting {wait}s …"
                    )
                    time.sleep(wait)
                    self._rate_limiter._calls.clear()
                    self._rate_limiter.wait()
                else:
                    raise
        raise RuntimeError(f"Gemini call failed after {retries} retries")

    def _call_vlm(self, pil_img: PILImage.Image, prompt: str) -> str:
        raw = self._call_gemini(pil_img, prompt)
        raw = _strip_code_fences(raw)
        raw = normalize_tables_in_markdown(raw)
        return raw

    # ── Helpers ───────────────────────────────────────────────────────────────
    def _is_complex_table(self, table: TableItem) -> bool:
        try:
            return (
                table.data.num_rows > self._complex_table_rows
                or table.data.num_cols > self._complex_table_cols
            )
        except Exception:
            return False


    def _count_pages(self, pdf_path: str) -> int:
        """Count PDF pages without loading into docling."""
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(pdf_path)
        count = len(pdf)
        pdf.close()
        return count

    @staticmethod
    def _item_sort_key(item) -> tuple[float, float]:
        """Return (y, x) sort key in top-left origin from docling provenance."""
        if not item.prov:
            return (0.0, 0.0)
        bbox = item.prov[0].bbox
        # Docling bbox: l/t/r/b in BOTTOMLEFT origin → t is top (largest y).
        # For sort order we want top-of-element first (largest bbox.t = closest to top).
        # Negate t so highest-on-page items sort first.
        return (-bbox.t, bbox.l)

    # ── Per-page assembly ─────────────────────────────────────────────────────

    def _find_same_band_items(self, anchor, all_items, h_gap_thresh: float = 20.0):
        """Return TextItems/SectionHeaderItems in the same horizontal Y-band as
        *anchor* (PictureItem) and horizontally adjacent within h_gap_thresh pts.
        These are merged into the anchor picture Gemini crop."""
        if not anchor.prov:
            return []
        ab = anchor.prov[0].bbox
        a_page = anchor.prov[0].page_no
        a_height = max(ab.t - ab.b, 1.0)
        result = []
        for item in all_items:
            if item is anchor:
                continue
            if not isinstance(item, (TextItem, SectionHeaderItem)):
                continue
            if not item.prov or item.prov[0].page_no != a_page:
                continue
            ib = item.prov[0].bbox
            v_overlap = min(ab.t, ib.t) - max(ab.b, ib.b)
            if v_overlap / a_height < 0.2:
                continue
            if ib.l > ab.r:
                h_gap = ib.l - ab.r
            elif ib.r < ab.l:
                h_gap = ab.l - ib.r
            else:
                h_gap = 0
            if h_gap <= h_gap_thresh:
                result.append(item)
        return result

    def _expand_and_crop(self, doc, page_no: int, bboxes, padding: int = 8):
        """Crop the union of *bboxes* from the page image (BOTTOMLEFT → PIL).
        At images_scale pts/px: pix = coord * images_scale."""
        page = doc.pages.get(page_no)
        if page is None or page.image is None or page.image.pil_image is None:
            return None
        pil_full = page.image.pil_image
        img_w, img_h = pil_full.size
        ph = page.size.height
        l = min(b.l for b in bboxes)
        bot = min(b.b for b in bboxes)
        r = max(b.r for b in bboxes)
        t = max(b.t for b in bboxes)
        sc = self._images_scale
        pix_l = max(0,     int(l * sc) - padding)
        pix_t = max(0,     int((ph - t) * sc) - padding)
        pix_r = min(img_w, int(r * sc) + padding)
        pix_b = min(img_h, int((ph - bot) * sc) + padding)
        if pix_r <= pix_l or pix_b <= pix_t:
            return None
        return pil_full.crop((pix_l, pix_t, pix_r, pix_b))

    def _process_page(self, page_no: int, items: list, doc) -> str:
        """Assemble one page markdown in reading order.

        Pre-pass: text items in the same horizontal band as a PictureItem are
        merged into its Gemini crop (expanded bbox) and skipped individually.
        This prevents adjacent styled text from being garbled by docling OCR.
        """
        # ── Pre-pass ─────────────────────────────────────────────────────────
        adjacent_texts: dict = {}
        skip_ids: set = set()
        for item in items:
            if not isinstance(item, PictureItem) or not item.prov:
                continue
            band = self._find_same_band_items(item, items)
            if band:
                adjacent_texts[id(item)] = band
                skip_ids.update(id(t) for t in band)

        # ── Main pass ─────────────────────────────────────────────────────────
        ordered: list = []

        for item in items:
            if not item.prov:
                continue
            y, x = self._item_sort_key(item)

            if id(item) in skip_ids:
                continue  # absorbed into adjacent picture Gemini crop

            # ── Image → docling crop → Gemini ────────────────────────────────
            if isinstance(item, PictureItem):
                adj = adjacent_texts.get(id(item))
                if adj:
                    bboxes = [item.prov[0].bbox] + \
                        [t.prov[0].bbox for t in adj]
                    pil = self._expand_and_crop(
                        doc, item.prov[0].page_no, bboxes)
                    if pil is None:
                        pil = item.get_image(doc)
                    label = f"image+{len(adj)} text-items"
                else:
                    pil = item.get_image(doc)
                    label = "image"
                if pil is None:
                    logger.warning(
                        f"  p{page_no}: PictureItem has no image, skipping")
                    continue
                logger.info(
                    f"  p{page_no}: {label} ({pil.width}×{pil.height}px) → Gemini")
                md = self._call_vlm(pil, VLM_IMAGE_PROMPT).strip()

            # ── Complex table → docling crop → Gemini ────────────────────────
            elif isinstance(item, TableItem) and self._is_complex_table(item):
                pil = item.get_image(doc)
                if pil is None:
                    logger.warning(
                        f"  p{page_no}: complex table has no image, falling back to Docling"
                    )
                    try:
                        md = item.export_to_markdown()
                    except Exception:
                        md = str(item.data)
                    md = f"<table>\n\n{md}\n\n</table>"
                else:
                    logger.info(
                        f"  p{page_no}: complex table "
                        f"({item.data.num_rows}×{item.data.num_cols}, "
                        f"{pil.width}×{pil.height}px) → Gemini"
                    )
                    md = self._call_vlm(pil, VLM_TABLE_PROMPT).strip()
                    if not md.startswith("<table>"):
                        md = f"<table>\n\n{md}\n\n</table>"

            # ── Simple table → Docling ────────────────────────────────────────
            elif isinstance(item, TableItem):
                logger.debug(
                    f"  p{page_no}: simple table "
                    f"({item.data.num_rows}×{item.data.num_cols}) → Docling"
                )
                try:
                    md = item.export_to_markdown()
                except Exception:
                    try:
                        md = item.export_to_dataframe().to_markdown(index=False)
                    except Exception:
                        md = str(item.data)
                md = f"<table>\n\n{md}\n\n</table>"

            # ── Section header → Docling ──────────────────────────────────────
            elif isinstance(item, SectionHeaderItem):
                level = "#" * min(max(item.level, 1), 6)
                md = f"{level} {item.text}"

            # ── Body text → Docling ───────────────────────────────────────────
            elif isinstance(item, TextItem):
                md = (item.text or "").strip()
                if not md:
                    continue

            else:
                continue

            ordered.append((y, x, md))

        ordered.sort(key=lambda t: (t[0], t[1]))
        body = "\n\n".join(md for _, _, md in ordered)
        return f"[PAGE:{page_no}]\n\n{body}"

    # ── Public API ────────────────────────────────────────────────────────────
    def parse_pdf(
        self,
        pdf_path: str,
        output_path: Optional[str] = None,
    ) -> str:
        """
        Parse a PDF to markdown. No fitz — image rendering is 100% via docling.

        Args:
            pdf_path:    Path to the PDF file.
            output_path: Optional output path; each page is flushed immediately.
        Returns:
            Full markdown string.
        """
        self._vlm_calls = 0
        pdf_path = str(pdf_path)

        logger.info(
            f"═══ HybridDoclingGeminiParser (fitz-free): {Path(pdf_path).name} ═══")

        # ── Step 1: Docling full-document conversion ───────────────────────────
        # generate_page_images + generate_picture_images enable image access
        # via item.get_image(doc) without any fitz calls.
        logger.info("Step 1/3  Docling converting "
                    "(OCR + layout + tables + page/picture image rendering) …")
        conv = self._build_converter().convert(pdf_path)
        doc = conv.document

        # ── Step 2: Group elements by page ────────────────────────────────────
        logger.info("Step 2/3  Grouping elements by page …")
        page_items: dict[int, list] = defaultdict(list)
        for item, _level in doc.iterate_items():
            if item.prov:
                page_items[item.prov[0].page_no].append(item)

        total_pages = len(doc.pages)
        total_pages = 20 # test on 20 pages
        total_items = sum(len(v) for v in page_items.values())
        logger.info(f"{total_pages} pages, {total_items} elements")

        # ── Step 3: Page-by-page assembly ─────────────────────────────────────
        logger.info("Step 3/3  Assembling pages …")
        pages_md = []
        out_file = None

        if output_path:
            Path(output_path).parent.mkdir(parents=True, exist_ok=True)
            out_file = open(output_path, "w", encoding="utf-8")

        try:
            for page_no in range(1, total_pages + 1):
                print(f"[{page_no}/{total_pages}] … ", end="", flush=True)
                try:
                    page_md = self._process_page(
                        page_no=page_no,
                        items=page_items.get(page_no, []),
                        doc=doc,
                    )
                    page_md = normalize_tables_in_markdown(page_md)
                    page_md = _clean_html(page_md)
                    page_md = _fix_table_closing_tags(page_md)

                    chunk = page_md + "\n\n---\n\n"
                    pages_md.append(chunk)

                    if out_file:
                        out_file.write(chunk)
                        out_file.flush()

                    print("done")
                except Exception as exc:
                    if out_file:
                        out_file.write(chunk)
                        out_file.flush()
                    print(f"ERROR: {exc}")
                    logger.error(f"[page {page_no}] {exc}", exc_info=True)
        finally:
            if out_file:
                out_file.close()

        markdown = "".join(pages_md)
        if output_path:
            logger.info(f"Saved → {output_path}")

        print(f"\nDone. Gemini calls: {self._vlm_calls} (cropped regions only)  "
              f"|  pages: {total_pages}")
        return markdown

    # ── Alternative: true page-by-page conversion ─────────────────────────────
    def parse_pdf_page_by_page(
        self,
        pdf_path: str,
        output_path: Optional[str] = None,
    ) -> str:
        """
        Alternative entry point: converts one page at a time using
        docling's page_range=(n, n) parameter.

        Trade-off: lower peak memory, but each convert() call re-initialises
        the docling pipeline, making this significantly slower than parse_pdf()
        for large documents. Use only when memory is the bottleneck.
        """
        self._vlm_calls = 0
        pdf_path = str(pdf_path)

        # Probe total page count with a minimal (no image) conversion
        total_pages = self._count_pages(pdf_path)
        logger.info(f"Total pages: {total_pages}")
        total_pages = 20
        logger.info(f"Total pages: {total_pages}")

        pages_md = []
        out_file = None
        if output_path:
            Path(output_path).parent.mkdir(parents=True, exist_ok=True)
            out_file = open(output_path, "w", encoding="utf-8")

        converter = self._build_converter()

        try:
            for page_no in range(1, total_pages + 1):
                print(f"[{page_no}/{total_pages}] converting … ",
                      end="", flush=True)
                try:
                    # Convert exactly one page — docling page_range is 1-based inclusive
                    conv = converter.convert(
                        pdf_path, page_range=(page_no, page_no))
                    doc = conv.document

                    items = []
                    for item, _level in doc.iterate_items():
                        if item.prov:
                            items.append(item)

                    page_md = self._process_page(
                        page_no=page_no,
                        items=items,
                        doc=doc,
                    )
                    page_md = normalize_tables_in_markdown(page_md)
                    page_md = _clean_html(page_md)
                    page_md = _fix_table_closing_tags(page_md)

                    chunk = page_md + "\n\n---\n\n"
                    pages_md.append(chunk)

                    if out_file:
                        out_file.write(chunk)
                        out_file.flush()

                    print("done")
                except Exception as exc:
                    print(f"ERROR: {exc}")
                    logger.error(f"[page {page_no}] {exc}", exc_info=True)
        finally:
            if out_file:
                out_file.close()

        markdown = "".join(pages_md)
        if output_path:
            logger.info(f"Saved → {output_path}")

        print(f"\nDone. Gemini calls: {self._vlm_calls} (cropped regions only)  "
              f"|  pages: {total_pages}")
        return markdown

c:\Users\vietl\.conda\envs\longnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
input_path = r"D:\Books\2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf"
output_path = r"output_test.md"

parser = HybridDoclingGeminiParser(
    api_key            = GOOGLE_API_KEY,
    gemini_model       = GEMINI_MODEL,
    rpm_limit          = GEMINI_RPM_LIMIT,
    complex_table_rows = COMPLEX_TABLE_ROWS,
    complex_table_cols = COMPLEX_TABLE_COLS,
)

md = parser.parse_pdf_page_by_page(
    pdf_path    = input_path,
    output_path = output_path,
)

print(f"\nOutput saved → {output_path}")
print(f"Total chars: {len(md)}")

INFO Total pages: 510
INFO Total pages: 20


[1/20] converting … 

INFO detected formats: [<InputFormat.PDF: 'pdf'>]
INFO Going to convert document batch...
INFO Initializing pipeline for StandardPdfPipeline with options hash c26cb460a55f99d5f30b76a2db1c7e20
INFO Accelerator device: 'cpu'
INFO Accelerator device: 'cpu'
INFO Processing document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf
INFO Finished converting document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf in 2.41 sec.
INFO   p1: image (135×25px) → Gemini
ERROR [page 1] [Gemini] Daily quota exhausted.
Traceback (most recent call last):
  File "C:\Users\vietl\AppData\Local\Temp\ipykernel_22244\3441219949.py", line 243, in _call_gemini
    return model.generate_content([pil_img, prompt]).text
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "c:\U

ERROR: [Gemini] Daily quota exhausted.
[2/20] converting … 

INFO detected formats: [<InputFormat.PDF: 'pdf'>]
INFO Going to convert document batch...
INFO Initializing pipeline for StandardPdfPipeline with options hash dc25705989614b708d59d464031af138
INFO Accelerator device: 'cpu'
INFO Accelerator device: 'cpu'
INFO Processing document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf
INFO Finished converting document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf in 1.61 sec.


...

ERROR: [Gemini] Daily quota exhausted.
[4/20] converting … 

INFO detected formats: [<InputFormat.PDF: 'pdf'>]
INFO Going to convert document batch...
INFO Processing document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf
INFO Finished converting document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf in 0.63 sec.


done
[5/20] converting … 

INFO detected formats: [<InputFormat.PDF: 'pdf'>]
INFO Going to convert document batch...
INFO Processing document 2-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf


KeyboardInterrupt: 

In [ ]:
# Merge 
from altair import value


intervals = [[1,3],[2,6],[7,9],[8,10],[15,18]]

class Solution:
    def merge(self, intervals):
        # Step 1: sort the current intervals and get the first item
        intervals.sort(key=lambda x: x[0]) 
        results = [intervals[0]]

        # Step 2 Loop through the interval
        for item in intervals:
            last = results[-1]
            if last[1] >= item[0]:
                last[1] = max(last[1], item[0]) 
            else:
                results.append(item)
        return results

a = Solution()
a.merge(intervals)


[[1, 3], [7, 9], [15, 18]]

In [14]:
# Balon problems
points = [[10,16],[2,8],[1,6],[7,12]]

class Solution:
    def findMinArrowShots(self, points):
        # Step 1: sort the intervals + get the initial arrow = 1
        points.sort(key=lambda x: x[1])

        arrow = 1
        current = points[0][1]
        
        # Step 2: For each ballon, compare its last value with the current ballon
        for ballon in points:
            if ballon[0] > current:
                arrow += 1
                current = ballon[1]

        return arrow


a = Solution()
a.findMinArrowShots(points)


2

In [8]:
# def grocery_store(A):
#     n = len(A)
#     size, result = 0, 0
#     for i in range(n):
#         if A[i] == 0:
#             size += 1
#         else:
#             size -= 1
#         result = max(result, -size)
#     return result

N = 5
queue = [0] * N
head, tail = 0, 0

def push(x):
    global tail
    tail = (tail + 1) % N
    queue[tail] = x

push(2)
queue
# grocery_store([1,0,0,1,1,1,1,0,1,0,1])

[0, 2, 0, 0, 0]